In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# データ分割
from sklearn.model_selection import train_test_split

# 線形モデル
from sklearn.ensemble import RandomForestClassifier

# グラフをアウトプット行に出力するためのマジックコマンド
%matplotlib inline

# 精度評価
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from lazypredict.Supervised import LazyClassifier 
import lightgbm as lgb
import optuna

In [2]:
train = pd.read_csv('../../data/processed/train_name_dropped.csv')
test = pd.read_csv('../../data/processed/test_name_dropped.csv')

In [3]:
# 説明変数と目的変数に分割

# 説明変数
X = train.drop(['Exited'],axis=1)
# 目的変数
y = train['Exited']

In [4]:
# データの分割
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    random_state=42,
                                                    stratify=y)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (11250, 19), X_test: (3750, 19)


## 複数のモデルでROCを比較

In [5]:
clf = LazyClassifier(ignore_warnings=True, predictions=True)   #設定
models, predictions = clf.fit(X_train, X_test, y_train, y_test) #実行

In [6]:
models

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
LinearDiscriminantAnalysis,0.890933,0.822606,0.922576,0.889861,0.889054,0.890933,0.136981
LGBMClassifier,0.896800,0.819972,0.931071,0.894430,0.893457,0.896800,0.367292
AdaBoostClassifier,0.898133,0.819352,0.927837,0.895470,0.894604,0.898133,0.877916
LogisticRegression,0.894400,0.814576,0.922640,0.891762,0.890754,0.894400,0.097028
NearestCentroid,0.802667,0.814313,0.904398,0.817708,0.859585,0.802667,0.108189
RandomForestClassifier,0.895467,0.814274,0.922226,0.892611,0.891694,0.895467,2.050874
BernoulliNB,0.840800,0.813962,0.889598,0.848332,0.863617,0.840800,0.096460
CalibratedClassifierCV,0.893867,0.813755,0.922928,0.891216,0.890191,0.893867,0.319107
LinearSVC,0.892800,0.810655,0.922903,0.889935,0.888915,0.892800,0.097107


## LightGBMを用いてモデルを作成する

In [7]:
dtrain = lgb.Dataset(X_train, y_train)
dvalid = lgb.Dataset(X_test, y_test)

params = {
    'objective': 'binary',
    'metrics': 'auc',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'learning_rate': 0.01   
}

model = lgb.train(params, dtrain, num_boost_round=1000)
pred_prob = model.predict(X_test)

## Optunaを用いてhyper parameterのチューニングを行う

In [8]:
# 目的関数の定義
def objective(trial):
    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_test, y_test)

    params = {
        'objective': 'binary',
        'metrics': 'auc',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'lambda_l1'         : trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2'         : trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),

        'force_col_wise':True,
        'random_state': 0,
    }

    model = lgb.train(
        params=params,
        train_set=dtrain,
        num_boost_round=100,
    )

    pred = model.predict(X_test, num_iteration=model.best_iteration)
    score = roc_auc_score(y_test, pred)
    
    return score

In [9]:
# 最適化の実行
optuna.logging.disable_default_handler()
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42), )
study.optimize(objective, n_trials=60)

In [10]:
# 最も精度の高いパラメータ
print('＝＝＝＝ベストパラメーター＝＝＝＝＝')
print(study.best_params)

# 最も精度の高いパラメータで学習を再度実行
best_params = {
        'objective': 'binary',
        'metrics': 'auc',
        'boosting_type': 'gbdt',
        'verbosity': -1
    }
best_params.update(study.best_params)

＝＝＝＝ベストパラメーター＝＝＝＝＝
{'lambda_l1': 5.11924119595392, 'lambda_l2': 2.2001736702106176, 'learning_rate': 0.05021823948534462, 'num_leaves': 77, 'feature_fraction': 0.6446457690396155, 'bagging_fraction': 0.976257447000586, 'bagging_freq': 1, 'min_child_samples': 65}


## 層化K分割を用いてクロスバリデーションを行い、モデルの性能を評価する

In [11]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=43)
valid_score = []

for fold, (train_index, valid_index) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_index], X.iloc[valid_index]
    y_tr, y_va = y.iloc[train_index], y.iloc[valid_index]
    set_tr = lgb.Dataset(X_tr, y_tr)
    
    model_tmp = lgb.train(
        params = best_params,
        train_set = set_tr
    )

    pred = model_tmp.predict(X_va)
    roc_score = roc_auc_score(y_va, pred)
    valid_score.append(roc_score)

    print(f'fold:{fold + 1} roc_score: {roc_score}')

cv_score = np.mean(valid_score)
print(f'cv_score: {cv_score}')

fold:1 roc_score: 0.9339398517642679
fold:2 roc_score: 0.9376946688781598
fold:3 roc_score: 0.9369888658981183
fold:4 roc_score: 0.9341068085525669
fold:5 roc_score: 0.9321094853351727
cv_score: 0.9349679360856571


In [12]:
best_model = lgb.train(
    params = best_params,
    train_set = dtrain
)

pd.DataFrame(best_model.feature_importance(), index=X_train.columns )

,0
CreditScore,651
Age,533
Tenure,225
Balance,375
EstimatedSalary,368
Geography_France,38
Geography_Germany,166
Geography_Spain,3
Gender_Female,149
Gender_Male,51


## 学習済みモデルを用いてテストデータを予測

In [13]:
pred_df = test.drop(['Exited'], axis = 1)
pred_test = best_model.predict(pred_df)